<a href="https://colab.research.google.com/github/maverick8675309/seaid-framework/blob/main/Notebooks/10_SEAID_Early_Warning_Framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SEAID Temporal Early-Warning Framework

## Purpose

This notebook operationalizes the predictive modeling and model-comparison results developed in SEAID Notebooks 01–09.

The modeling pipeline evaluates five approaches across Days 7, 14, 21, and 30:

- Logistic Regression
- Decision Tree
- Random Forest
- XGBoost
- Neural Network

Notebook 09 consolidates their temporal performance. This notebook then demonstrates how a selected, deployment-compatible model can translate predicted probabilities into longitudinal student-risk profiles, intervention tiers, and advisor-facing summaries.

The current operational implementation uses the saved XGBoost checkpoint models because they provide `predict_proba`, are available as complete joblib pipelines, and can be applied consistently at all four temporal checkpoints. The neural network remains an important deep-learning benchmark in the model-comparison stage.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/SEAID_Framework"
)

PROCESSED_DATA_DIR = (
    PROJECT_DIR /
    "data" /
    "processed"
)

NOTEBOOK_DIR = (
    PROJECT_DIR /
    "notebooks"
)

MODEL_DIR = (
    PROJECT_DIR /
    "models"
)

OUTPUTS_DIR = (
    PROJECT_DIR /
    "outputs"
)

RESULTS_DIR = (
    PROJECT_DIR /
    "results"
)

FIGURE_DIR = (
    PROJECT_DIR /
    "figures"
)

MODEL_COMPARISON_PATH = (
    RESULTS_DIR /
    "all_models_temporal_comparison.csv"
)

for directory in [
    MODEL_DIR,
    OUTPUTS_DIR,
    RESULTS_DIR,
    FIGURE_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print("Project directory:", PROJECT_DIR)
print("Model comparison file:", MODEL_COMPARISON_PATH)


In [ ]:
import joblib
import pandas as pd
import numpy as np

## Verify the Model-Comparison Recommendation

Notebook 09 saves a consolidated temporal comparison file. The following cells summarize mean performance across checkpoints and identify the leading model by mean ROC-AUC.

This verification does not silently switch the operational engine. A different model can be deployed only after its four checkpoint models and preprocessing objects have been exported in a compatible prediction format.


In [ ]:
if MODEL_COMPARISON_PATH.exists():

    model_comparison_df = pd.read_csv(
        MODEL_COMPARISON_PATH
    )

    required_comparison_columns = {
        "Model",
        "Checkpoint Day",
        "ROC-AUC"
    }

    missing_comparison_columns = (
        required_comparison_columns
        - set(model_comparison_df.columns)
    )

    if missing_comparison_columns:
        print(
            "Model comparison file is missing columns:",
            sorted(missing_comparison_columns)
        )
        model_ranking = pd.DataFrame()

    else:
        model_ranking = (
            model_comparison_df
            .groupby("Model", as_index=False)
            .agg(
                Mean_ROC_AUC=("ROC-AUC", "mean"),
                Mean_Accuracy=("Accuracy", "mean"),
                Mean_F1=("F1 Score", "mean"),
                Mean_Recall_Unsuccessful=(
                    "Recall - Unsuccessful",
                    "mean"
                )
            )
            .sort_values(
                "Mean_ROC_AUC",
                ascending=False
            )
            .reset_index(drop=True)
        )

        display(model_ranking)

        comparison_leader = (
            model_ranking
            .iloc[0]["Model"]
        )

        print(
            "Leading model by mean ROC-AUC:",
            comparison_leader
        )

        if comparison_leader != "XGBoost":
            print(
                "\nDeployment note: the operational cells below "
                "still use XGBoost because the four saved XGBoost "
                "pipelines are available in a common predict_proba "
                "format. Export equivalent checkpoint artifacts "
                f"before replacing XGBoost with {comparison_leader}."
            )

else:
    model_comparison_df = pd.DataFrame()
    model_ranking = pd.DataFrame()

    print(
        "Model comparison file not found. "
        "Run Notebook 09 first. The framework will continue "
        "using the saved XGBoost checkpoint models."
    )


## Save the Operational Model Ranking

When Notebook 09 results are available, the framework saves the aggregate ranking used to verify the operational model choice.


In [ ]:
if not model_ranking.empty:

    OPERATIONAL_RANKING_PATH = (
        RESULTS_DIR /
        "operational_model_ranking.csv"
    )

    model_ranking.to_csv(
        OPERATIONAL_RANKING_PATH,
        index=False
    )

    print(
        "Saved operational model ranking:",
        OPERATIONAL_RANKING_PATH
    )
else:
    print(
        "Operational model ranking was not saved "
        "because Notebook 09 results were unavailable."
    )


In [ ]:
DAY7_MODEL = MODEL_DIR / "xgboost_day7.joblib"
DAY14_MODEL = MODEL_DIR / "xgboost_day14.joblib"
DAY21_MODEL = MODEL_DIR / "xgboost_day21.joblib"
DAY30_MODEL = MODEL_DIR / "xgboost_day30.joblib"

XGBOOST_MODEL_PATHS = {
    7: DAY7_MODEL,
    14: DAY14_MODEL,
    21: DAY21_MODEL,
    30: DAY30_MODEL
}

missing_model_files = [
    str(model_path)
    for model_path in XGBOOST_MODEL_PATHS.values()
    if not model_path.exists()
]

if missing_model_files:
    raise FileNotFoundError(
        "Missing XGBoost checkpoint models. "
        "Run Notebook 07 first. Missing files:\n"
        + "\n".join(missing_model_files)
    )

for checkpoint_day, model_path in XGBOOST_MODEL_PATHS.items():
    print(
        f"Day {checkpoint_day} model:",
        model_path
    )


In [ ]:
xgboost_models = {
    checkpoint_day: joblib.load(model_path)
    for checkpoint_day, model_path
    in XGBOOST_MODEL_PATHS.items()
}

xgb_day7 = xgboost_models[7]
xgb_day14 = xgboost_models[14]
xgb_day21 = xgboost_models[21]
xgb_day30 = xgboost_models[30]

print(
    "All four XGBoost checkpoint models loaded successfully."
)


In [ ]:
TEMPORAL_DATASET_PATHS = {
    7: (
        PROCESSED_DATA_DIR /
        "early_warning_day7_dataset.csv"
    ),
    14: (
        PROCESSED_DATA_DIR /
        "early_warning_day14_dataset.csv"
    ),
    21: (
        PROCESSED_DATA_DIR /
        "early_warning_day21_dataset.csv"
    ),
    30: (
        PROCESSED_DATA_DIR /
        "early_warning_day30_dataset.csv"
    )
}

missing_dataset_files = [
    str(dataset_path)
    for dataset_path in TEMPORAL_DATASET_PATHS.values()
    if not dataset_path.exists()
]

if missing_dataset_files:
    raise FileNotFoundError(
        "Missing temporal datasets. "
        "Run Notebook 02 first. Missing files:\n"
        + "\n".join(missing_dataset_files)
    )

temporal_datasets = {
    checkpoint_day: pd.read_csv(dataset_path)
    for checkpoint_day, dataset_path
    in TEMPORAL_DATASET_PATHS.items()
}

day7_df = temporal_datasets[7]
day14_df = temporal_datasets[14]
day21_df = temporal_datasets[21]
day30_df = temporal_datasets[30]

print("Temporal datasets loaded successfully.")


In [ ]:
print(day7_df.shape)
print(day14_df.shape)
print(day21_df.shape)
print(day30_df.shape)

In [ ]:
TARGET = "target_success"

print(day7_df[TARGET].value_counts())

In [ ]:
def inspect_model_features(model, model_name):
    print(f"\n{model_name}")

    if hasattr(model, "feature_names_in_"):
        print(
            "Expected features:",
            len(model.feature_names_in_)
        )

        print(
            list(model.feature_names_in_)[:10]
        )
    else:
        print(
            "No feature_names_in_ attribute found."
        )


inspect_model_features(
    xgb_day7,
    "Day 7 XGBoost"
)

inspect_model_features(
    xgb_day14,
    "Day 14 XGBoost"
)

inspect_model_features(
    xgb_day21,
    "Day 21 XGBoost"
)

inspect_model_features(
    xgb_day30,
    "Day 30 XGBoost"
)

In [ ]:
print(day7_df.columns.tolist())

In [ ]:
TARGET = "target_success"

POSSIBLE_ID_COLUMNS = [
    "id_student",
    "code_module",
    "code_presentation"
]

In [ ]:
def prepare_model_features(
    dataframe,
    model,
    target_column="target_success"
):
    if hasattr(model, "feature_names_in_"):

        expected_features = list(
            model.feature_names_in_
        )

        missing_features = [
            column
            for column in expected_features
            if column not in dataframe.columns
        ]

        if missing_features:
            raise ValueError(
                "Missing required features: "
                f"{missing_features}"
            )

        X = dataframe[
            expected_features
        ].copy()

    else:
        excluded_columns = [
            target_column
        ]

        X = dataframe.drop(
            columns=[
                column
                for column in excluded_columns
                if column in dataframe.columns
            ]
        ).copy()

    return X

In [ ]:
X_day7 = prepare_model_features(
    day7_df,
    xgb_day7
)

X_day14 = prepare_model_features(
    day14_df,
    xgb_day14
)

X_day21 = prepare_model_features(
    day21_df,
    xgb_day21
)

X_day30 = prepare_model_features(
    day30_df,
    xgb_day30
)

In [ ]:
print("Day 7:", X_day7.shape)
print("Day 14:", X_day14.shape)
print("Day 21:", X_day21.shape)
print("Day 30:", X_day30.shape)

In [ ]:
print("Day 7 classes:", xgb_day7.classes_)
print("Day 14 classes:", xgb_day14.classes_)
print("Day 21 classes:", xgb_day21.classes_)
print("Day 30 classes:", xgb_day30.classes_)

In [ ]:
day7_risk_probability = (
    xgb_day7.predict_proba(X_day7)[:, 0]
)

day14_risk_probability = (
    xgb_day14.predict_proba(X_day14)[:, 0]
)

day21_risk_probability = (
    xgb_day21.predict_proba(X_day21)[:, 0]
)

day30_risk_probability = (
    xgb_day30.predict_proba(X_day30)[:, 0]
)

In [ ]:
print(
    "Day 7 risk range:",
    day7_risk_probability.min(),
    day7_risk_probability.max()
)

print(
    "Day 14 risk range:",
    day14_risk_probability.min(),
    day14_risk_probability.max()
)

print(
    "Day 21 risk range:",
    day21_risk_probability.min(),
    day21_risk_probability.max()
)

print(
    "Day 30 risk range:",
    day30_risk_probability.min(),
    day30_risk_probability.max()
)

In [ ]:
def assign_risk_tier(probability):

    if probability >= 0.75:
        return "Critical"

    elif probability >= 0.50:
        return "High"

    elif probability >= 0.25:
        return "Moderate"

    else:
        return "Low"

In [ ]:
def build_prediction_table(
    dataframe,
    risk_probability,
    checkpoint
):

    prediction_df = pd.DataFrame()

    for column in [
        "id_student",
        "code_module",
        "code_presentation"
    ]:

        if column in dataframe.columns:
            prediction_df[column] = dataframe[column]

    prediction_df["Checkpoint"] = checkpoint

    prediction_df["Risk Probability"] = risk_probability

    prediction_df["Risk Tier"] = (
        prediction_df["Risk Probability"]
        .apply(assign_risk_tier)
    )

    prediction_df["Predicted Outcome"] = np.where(
        prediction_df["Risk Probability"] >= 0.50,
        "At Risk",
        "Likely Successful"
    )

    if TARGET in dataframe.columns:
        prediction_df["Actual Outcome"] = np.where(
            dataframe[TARGET] == 1,
            "Successful",
            "Unsuccessful"
        )

    return prediction_df

In [ ]:
predictions_day7 = build_prediction_table(
    day7_df,
    day7_risk_probability,
    7
)

predictions_day14 = build_prediction_table(
    day14_df,
    day14_risk_probability,
    14
)

predictions_day21 = build_prediction_table(
    day21_df,
    day21_risk_probability,
    21
)

predictions_day30 = build_prediction_table(
    day30_df,
    day30_risk_probability,
    30
)

In [ ]:
display(predictions_day7.head())

print("\nRisk Tier Counts\n")

print(
    predictions_day7["Risk Tier"]
    .value_counts()
)

In [ ]:
all_predictions = pd.concat(
    [
        predictions_day7,
        predictions_day14,
        predictions_day21,
        predictions_day30
    ],
    ignore_index=True
)

print(all_predictions.shape)

display(all_predictions.head())

In [ ]:
PREDICTION_FILE = (
    RESULTS_DIR /
    "student_risk_predictions.csv"
)

all_predictions.to_csv(
    PREDICTION_FILE,
    index=False
)

print("Saved:", PREDICTION_FILE)

In [ ]:
risk_tracker = (
    all_predictions
    .pivot_table(
        index=[
            "id_student",
            "code_module",
            "code_presentation"
        ],
        columns="Checkpoint",
        values="Risk Probability"
    )
    .reset_index()
)

risk_tracker.columns = [
    "id_student",
    "code_module",
    "code_presentation",
    "Day7",
    "Day14",
    "Day21",
    "Day30"
]

display(risk_tracker.head())

In [ ]:
risk_tracker["Change_7_to_30"] = (
    risk_tracker["Day30"]
    - risk_tracker["Day7"]
)

risk_tracker["Maximum_Risk"] = (
    risk_tracker[
        [
            "Day7",
            "Day14",
            "Day21",
            "Day30"
        ]
    ]
    .max(axis=1)
)

risk_tracker["Average_Risk"] = (
    risk_tracker[
        [
            "Day7",
            "Day14",
            "Day21",
            "Day30"
        ]
    ]
    .mean(axis=1)
)

In [ ]:
def classify_trend(row):

    if row["Day30"] >= 0.75:
        return "Persistently Critical"

    elif row["Change_7_to_30"] >= 0.20:
        return "Increasing Risk"

    elif row["Change_7_to_30"] <= -0.20:
        return "Improving"

    else:
        return "Stable"


risk_tracker["Risk Trend"] = (
    risk_tracker.apply(
        classify_trend,
        axis=1
    )
)

In [ ]:
print(
    risk_tracker["Risk Trend"]
    .value_counts()
)

display(risk_tracker.head())

In [ ]:
def intervention_level(row):

    if row["Day30"] >= 0.90:
        return "Immediate Intervention"

    elif row["Day30"] >= 0.75:
        return "Advisor Outreach"

    elif row["Average_Risk"] >= 0.50:
        return "Faculty Check-In"

    elif row["Average_Risk"] >= 0.25:
        return "Continue Monitoring"

    else:
        return "No Action Needed"


risk_tracker["Recommended Action"] = (
    risk_tracker.apply(
        intervention_level,
        axis=1
    )
)

In [ ]:
print(
    risk_tracker["Recommended Action"]
    .value_counts()
)

In [ ]:
dashboard = (
    risk_tracker
    .groupby("Recommended Action")
    .agg(
        Students=("id_student", "count"),
        Mean_Risk=("Average_Risk", "mean"),
        Mean_Day30_Risk=("Day30", "mean")
    )
    .sort_values(
        "Mean_Day30_Risk",
        ascending=False
    )
)

display(dashboard)

In [ ]:
risk_tracker.to_csv(
    RESULTS_DIR /
    "student_risk_tracker.csv",
    index=False
)

dashboard.to_csv(
    RESULTS_DIR /
    "intervention_dashboard.csv",
    index=False
)

print("Framework outputs saved.")

In [ ]:
import matplotlib.pyplot as plt

action_counts = (
    risk_tracker["Recommended Action"]
    .value_counts()
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 6))

action_counts.plot(kind="bar")

plt.title("SEAID Recommended Student Interventions")

plt.xlabel("Recommended Action")

plt.ylabel("Number of Students")

plt.xticks(rotation=20)

plt.tight_layout()

plt.savefig(
    FIGURES_DIR /
    "recommended_interventions.png",
    dpi=300
)

plt.show()

In [ ]:
action_order = [
    "Immediate Intervention",
    "Advisor Outreach",
    "Faculty Check-In",
    "Continue Monitoring",
    "No Action Needed"
]

action_counts = (
    risk_tracker["Recommended Action"]
    .value_counts()
    .reindex(action_order)
)

plt.figure(figsize=(10,6))

action_counts.plot(kind="bar")

plt.title("SEAID Framework: Recommended Student Interventions")

plt.xlabel("Intervention Level")

plt.ylabel("Number of Students")

plt.xticks(rotation=20)

plt.tight_layout()

plt.savefig(
    FIGURES_DIR /
    "recommended_interventions.png",
    dpi=300
)

plt.show()

In [ ]:
ax = action_counts.plot(kind="bar", figsize=(10,6))

plt.title("SEAID Framework: Recommended Student Interventions")
plt.xlabel("Intervention Level")
plt.ylabel("Number of Students")

for i, value in enumerate(action_counts):
    ax.text(
        i,
        value + 150,
        f"{value:,}",
        ha="center",
        fontsize=10
    )

plt.xticks(rotation=20)
plt.tight_layout()

plt.savefig(
    FIGURES_DIR /
    "recommended_interventions.png",
    dpi=300
)

plt.show()

In [ ]:
average_risk = pd.DataFrame({
    "Checkpoint": [7, 14, 21, 30],
    "Average Risk": [
        risk_tracker["Day7"].mean(),
        risk_tracker["Day14"].mean(),
        risk_tracker["Day21"].mean(),
        risk_tracker["Day30"].mean()
    ]
})

display(average_risk)

In [ ]:
plt.figure(figsize=(9,5))

plt.plot(
    average_risk["Checkpoint"],
    average_risk["Average Risk"],
    marker="o",
    linewidth=3
)

plt.title(
    "Average Predicted Student Risk Across Early-Warning Checkpoints"
)

plt.xlabel("Checkpoint (Day)")
plt.ylabel("Average Risk Probability")

plt.xticks([7,14,21,30])

plt.grid(alpha=.3)

plt.tight_layout()

plt.savefig(
    FIGURES_DIR /
    "average_risk_over_time.png",
    dpi=300
)

plt.show()

In [ ]:
risk_tiers = pd.DataFrame({
    "id_student": predictions_day7["id_student"],
    "code_module": predictions_day7["code_module"],
    "code_presentation": predictions_day7["code_presentation"],
    "Day7": predictions_day7["Risk Tier"],
    "Day14": predictions_day14["Risk Tier"],
    "Day21": predictions_day21["Risk Tier"],
    "Day30": predictions_day30["Risk Tier"]
})

display(risk_tiers.head())

In [ ]:
transition_matrix = pd.crosstab(
    risk_tiers["Day7"],
    risk_tiers["Day30"]
)

display(transition_matrix)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))

plt.imshow(
    transition_matrix,
    aspect="auto"
)

plt.colorbar(label="Students")

plt.xticks(
    range(len(transition_matrix.columns)),
    transition_matrix.columns
)

plt.yticks(
    range(len(transition_matrix.index)),
    transition_matrix.index
)

plt.xlabel("Day 30 Risk Tier")
plt.ylabel("Day 7 Risk Tier")

plt.title("Student Risk Tier Transitions (Day 7 → Day 30)")

plt.tight_layout()

plt.savefig(
    FIGURES_DIR /
    "risk_transition_matrix.png",
    dpi=300
)

plt.show()

In [ ]:
risk_order = {
    "Low": 1,
    "Moderate": 2,
    "High": 3,
    "Critical": 4
}

transition_analysis = risk_tiers.copy()

transition_analysis["Day7Score"] = (
    transition_analysis["Day7"].map(risk_order)
)

transition_analysis["Day30Score"] = (
    transition_analysis["Day30"].map(risk_order)
)

transition_analysis["Change"] = (
    transition_analysis["Day30Score"]
    - transition_analysis["Day7Score"]
)

transition_analysis["Status"] = np.where(
    transition_analysis["Change"] > 0,
    "Higher Risk",
    np.where(
        transition_analysis["Change"] < 0,
        "Lower Risk",
        "No Change"
    )
)

print(
    transition_analysis["Status"]
    .value_counts()
)

In [ ]:
transition_matrix.to_csv(
    RESULTS_DIR /
    "day7_to_day30_transition_matrix.csv"
)

transition_analysis.to_csv(
    RESULTS_DIR /
    "student_risk_transition_analysis.csv",
    index=False
)

print("Transition outputs saved.")

## Framework Summary

Notebook 10 operationalized the SEAID temporal early-warning framework after the five-model comparison completed in Notebook 09.

The broader SEAID modeling stage compared Logistic Regression, Decision Tree, Random Forest, XGBoost, and Neural Network models across Days 7, 14, 21, and 30. This notebook uses the saved XGBoost checkpoint pipelines as the current operational engine because they can generate consistent student-level probabilities at each checkpoint.

The framework converts those probabilities into:

- checkpoint-specific risk estimates,
- four risk tiers,
- longitudinal changes in risk,
- intervention recommendations,
- advisor-facing summary tables,
- transition analyses from Day 7 to Day 30, and
- exportable institutional decision-support files.

Model predictions are intended to support human judgment rather than replace it.


## Interpretation of Longitudinal Risk Patterns

The longitudinal analysis showed that most students were classified as stable across the four early-warning checkpoints. However, stability did not necessarily indicate low risk. Some students remained stable at elevated risk levels, while others remained consistently low risk.

A substantial group of students was classified as persistently critical, indicating that their predicted probability of unsuccessful completion remained very high by Day 30. These students represent the highest-priority population for intensive and coordinated intervention.

A smaller group demonstrated increasing risk between Days 7 and 30. Although this group was less numerous, it is operationally important because these students may not have appeared highly vulnerable at the earliest checkpoint.

Students classified as improving showed a meaningful reduction in predicted risk over time. In a live implementation, this pattern could be used to evaluate whether outreach, advising, instructional support, or changes in student engagement were associated with improved risk trajectories.

## Limitations

Several limitations should be considered when interpreting the framework.

First, the predictions are based on historical OULAD data and may not generalize directly to other institutions, student populations, course structures, or learning-management systems.

Second, the current operational implementation uses XGBoost checkpoint models even though Notebook 09 compares five modeling approaches. Operational deployment of another model requires equivalent saved preprocessing and checkpoint-specific prediction artifacts.

Third, the risk tiers and intervention thresholds are illustrative rather than institutionally validated. Predicted probabilities should be calibrated and thresholds should be selected in consultation with advisors, faculty, institutional researchers, and student-support professionals.

Fourth, model predictions may reflect structural inequalities contained in historical data. Strong predictive performance does not establish fairness, causality, or the appropriateness of every proposed intervention.

Finally, the framework estimates risk but does not demonstrate that a particular intervention causes improved student outcomes. Intervention effectiveness requires prospective evaluation.


## Ethical Use Principles

The SEAID Framework should be implemented according to the following principles:

1. Predictions should be used to offer support, not restrict opportunity.
2. Students should not be stigmatized or labeled solely on the basis of a risk score.
3. Access to student-level predictions should be limited to authorized personnel with a legitimate educational need.
4. Institutions should regularly evaluate model performance across student groups.
5. Intervention decisions should include human review and contextual information.
6. Risk thresholds should be monitored and adjusted based on institutional capacity and student outcomes.
7. Personally identifiable information should be protected according to applicable privacy requirements and institutional policy.

## Future Work

Future development of the SEAID Framework should include:

- SHAP-based global and student-level explanations for the selected operational model,
- direct comparison of calibration across all five models,
- fairness analysis across demographic and educational groups,
- uncertainty and confidence measures for individual predictions,
- validation on additional institutions and course settings,
- advisor feedback and usability testing,
- configurable intervention thresholds based on institutional capacity,
- deployment-compatible exports for the neural network and other alternative models,
- automated model monitoring and drift detection, and
- evaluation of whether interventions improve student outcomes.


## Operational Threshold Selection

The intervention thresholds used in this framework were selected to demonstrate how predictive probabilities can be translated into actionable decision-support categories. The thresholds are intended as illustrative examples rather than institutionally validated decision rules.

In practice, institutions should determine appropriate probability thresholds based on factors such as:

- institutional retention goals,
- advising and student support capacity,
- acceptable false-positive and false-negative rates,
- intervention costs,
- and historical intervention effectiveness.

Threshold optimization should therefore be considered an institution-specific implementation step rather than a fixed component of the predictive model.

## Conclusion

Notebook 10 transformed the SEAID model-comparison results into an operational temporal early-warning framework.

The completed workflow now supports:

- student-level risk estimation at Days 7, 14, 21, and 30,
- longitudinal risk tracking,
- early identification of increasing or persistent risk,
- intervention prioritization,
- advisor-facing summary outputs,
- reproducible exported files, and
- explicit linkage between model comparison and deployment.

The framework currently operationalizes the saved XGBoost checkpoint models while retaining the neural network as the deep-learning comparison benchmark. Any future change in the deployed model should be supported by validated checkpoint artifacts, comparable probability outputs, calibration analysis, explainability, fairness testing, and institutional review.

SEAID is a decision-support framework, not an automated decision-making system. Its predictions should be combined with advisor expertise, student context, transparent communication, and meaningful opportunities for human review.


In [ ]:
print("Notebook 10 result files:\n")

expected_result_files = [
    "student_risk_predictions.csv",
    "student_risk_tracker.csv",
    "intervention_dashboard.csv",
    "day7_to_day30_transition_matrix.csv",
    "student_risk_transition_analysis.csv",
    "operational_model_ranking.csv"
]

for file_name in expected_result_files:

    file_path = RESULTS_DIR / file_name

    print(
        f"{file_name}:",
        "FOUND" if file_path.exists() else "NOT CREATED"
    )
